In [1]:
import os
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.env_checker import check_env as check_env_sb3_compatibility
from stable_baselines3.common.monitor import Monitor
import torch

# Custom code
from environment.custom_env import SB3LuxEnvBase2
from environment.observation_wrapper import FlattenNormalizeObservation
from environment.action_wrapper import FlattenActionWrapper

# Reload packages without having to restart kernel
%load_ext autoreload
%autoreload 2

In [2]:
device = torch.device("cpu") # SB3 PPO doesn't really support GPU acceleration

In [3]:
random_seed = 42

In [4]:
def create_env(r_seed=42):
    base_env = SB3LuxEnvBase2(random_seed=r_seed)
    observation_wrapped_env = FlattenNormalizeObservation(base_env)
    fully_wrapped_env = FlattenActionWrapper(observation_wrapped_env)
    return fully_wrapped_env

In [23]:
check_env_sb3_compatibility(create_env())

In [24]:
# Create env and verify SB3 compatibility
num_envs = 8
env = make_vec_env(lambda:create_env(random_seed), n_envs=num_envs, seed=random_seed, vec_env_cls=SubprocVecEnv)

In [25]:
# --- PPO Agent Setup ---
log_dir = "../training_logs/"
model_dir = "../saved_models/"
lstm_model_path = os.path.join(model_dir, "ppo_lstm")
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# PPO Hyperparameters
# See SB3 documentation for more details: https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html
ppo_lstm_hyperparams = {
    'policy':"MlpLstmPolicy",
    'env': env,
    'learning_rate': 0.00025,
    'n_steps': 2048,
    'batch_size': 64,
    'n_epochs': 5,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'clip_range_vf': None,
    'normalize_advantage': True,
    'ent_coef': 0.001,
    'vf_coef': 0.5,
    'max_grad_norm': 0.5,
    'stats_window_size': 100,
    'device': device,
    'verbose': 0,
    'seed': random_seed,
}

# Load model if it exists, else create new model.
if os.path.isfile(lstm_model_path):
    model = RecurrentPPO.load(lstm_model_path, env=env, device=device)
else:
    model = RecurrentPPO(**ppo_lstm_hyperparams)

# Evaluate the agent periodically on a separate instance of the env
eval_env = make_vec_env(lambda: Monitor(create_env()), n_envs=1, seed=random_seed + 1, vec_env_cls=SubprocVecEnv)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=model_dir + 'best_model/',
    log_path=log_dir + 'eval_results/',
    eval_freq=max(5000 // (1 if 'num_envs' not in locals() else num_envs), 1),
    deterministic=True,
    render=False,
    n_eval_episodes=5
)

In [26]:
# Training
total_timesteps = 100_000
print(f"Starting PPO (LSTM) training for {total_timesteps} timesteps...")
model.learn(
    total_timesteps=total_timesteps,
    callback=[eval_callback],
    tb_log_name="PPO_LSTM_LuxAI",
    reset_num_timesteps=False
)
print("Training finished.")

Starting PPO (LSTM) training for 100000 timesteps...
Eval num_timesteps=5000, episode_reward=778.01 +/- 527.48
Episode length: 505.00 +/- 0.00
New best mean reward!
Eval num_timesteps=10000, episode_reward=441.84 +/- 408.86
Episode length: 505.00 +/- 0.00
Eval num_timesteps=15000, episode_reward=613.18 +/- 427.96
Episode length: 505.00 +/- 0.00
Eval num_timesteps=20000, episode_reward=377.06 +/- 443.58
Episode length: 505.00 +/- 0.00
Eval num_timesteps=25000, episode_reward=277.06 +/- 197.00
Episode length: 505.00 +/- 0.00
Eval num_timesteps=30000, episode_reward=154.59 +/- 143.26
Episode length: 505.00 +/- 0.00
Eval num_timesteps=35000, episode_reward=250.50 +/- 282.67
Episode length: 505.00 +/- 0.00
Eval num_timesteps=40000, episode_reward=60.35 +/- 110.40
Episode length: 505.00 +/- 0.00
Eval num_timesteps=45000, episode_reward=88.71 +/- 195.44
Episode length: 505.00 +/- 0.00
Eval num_timesteps=50000, episode_reward=344.34 +/- 276.70
Episode length: 505.00 +/- 0.00
Eval num_timesteps

In [27]:
# Saving and cleanup
model.save(lstm_model_path)
print(f"Final model saved to {lstm_model_path}")
env.close()
eval_env.close()

Final model saved to ../saved_models/ppo_lstm


2025-04-26 16:38:16,386	WARNING compression.py:16 -- lz4 not available, disabling sample compression. This will significantly impact RLlib performance. To install lz4, run `pip install lz4`.
